# 10 — Sensitivity analysis: the same rule with causal-discovery algorithms built for mixed data

## Why this notebook exists (disclosure — keep this text with the notebook)
Notebooks 06 (CDC) and 08 (NHANES) are the **pre-specified analysis**: DirectLiNGAM, the candidate rule,
and the observed-vs-expected (O/E) check, fixed before any results were seen. They are not changed.

This notebook is a **second round, decided after** reviewing the results and the figures of notebook 09
(October 21, 2026). Two things prompted it:
1. Stage 0's own assumption gate rated **both** datasets BOUNDED because most variables are binary
   (68% CDC, 72% NHANES). DirectLiNGAM infers structure from the shape of *continuous* noise, so the gate
   was already warning that its edges may be unreliable on these data.
2. On NHANES, the pre-specified rule missed the group with the most missed diagnoses ("no doctor visit"),
   because DirectLiNGAM linked that variable directly to the label, and it flagged one group that turned
   out not to be elevated ("smoker").

**What was seen before this notebook was written:** the NHANES answer key (which groups truly have more
missed diagnoses); all results of notebooks 06, 08 and 09; and one quick, single-run, non-bootstrapped trial
of PC-G² (8,000 rows) and LiM (1,500 rows) on the NHANES training split, which suggested the label's and the
cholesterol check's neighbors differ from DirectLiNGAM's. (That trial standardized the binary variables
before LiM, which LiM does not expect; it is done correctly below.)

**What is fixed before this notebook is run (do not change after seeing results):**
* Algorithms: DirectLiNGAM (the saved bootstraps from 06/08, unchanged), **PC with the G² test** on
  discretized data (causal-learn), and **LiM**, the LiNGAM extension for mixed continuous and binary data
  (lingam package; Zeng et al., CLeaR 2022).
* The same training rows, the same 50 bootstrap resamples (same seed), the same 80% stability threshold,
  adjacency only (both gates are BOUNDED), the same candidate rule, the same O/E check and the same
  held-out NHANES truth scoring as notebook 08.
* Every algorithm's result is reported on both datasets, whether or not it helps.
* CDC has no ground truth and nobody has seen an answer key for it, so it is the uncontaminated check:
  the question there is only whether the flag lists agree.

**How it is run (updated Oct 21, ~5:40 pm, before any full-size result existed):** the first launch ran the
resamples one after another; PC-G² took 27 minutes for a single full-size resample (about 23 hours per
dataset), so it was stopped before any result was produced. A reduced run with 5,000 rows per resample was then
done (notebook 10b; its results were seen). This notebook keeps the **original full-size design** — the same
discovery rows DirectLiNGAM used (up to 20,000) — and spreads the 50 resamples over the laptop's processor
cores, saving progress after each batch so an interrupted run resumes. Limiting PC's search depth to speed it
up was tested and rejected, because it changes which links are found. Both runs (10 and 10b) are reported.

| Section | Output |
|---|---|
| 1 Settings and data | — |
| 2 Bootstrap PC-G² and LiM | `results/10_edges_{algorithm}_{dataset}.csv` |
| 3 Neighbors of the label and the recording variable, side by side | `results/10_neighbors.csv` |
| 4 Candidate rule + O/E check per algorithm | `results/10_flags.csv` |
| 5 NHANES: flags against the blood-test truth | `results/10_truth_shortlists.csv` |
| 6 Summary | `results/10_summary.csv` |

## 1. Settings and data
`QUICK = True` only checks that the notebook runs (5 bootstraps, small samples); use `False` for the paper.

In [1]:
QUICK = False

import sys, time, json, io, contextlib, importlib.util, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import li_lib as li
warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 20261020                     # same seed as notebooks 06 and 08
BOOT = 5 if QUICK else 50           # same number of bootstrap resamples as Stage 0
MAX_N = 20000                       # same discovery-sample cap as 06/08
LIM_MAX_N = 2000 if QUICK else MAX_N   # full discovery sample, the same rows as DirectLiNGAM
PC_MAX_N = 2000 if QUICK else MAX_N    # (PC-G2 full size: about 27 min per resample on one core)
N_JOBS = li.N_JOBS                  # resamples run in parallel on this many processor cores
EDGE_P = 0.80                       # stable = adjacent in >= 80% of bootstraps
PC_ALPHA = 0.01                     # same alpha as the PC cross-check in Stage 0
N_BINS = 5                          # PC-G2: variables with more than 5 values are cut into fifths
MIN_SHARE, MAX_SHARE, MIN_N_TRAIN = 0.03, 0.50, 500
ALPHA, N_TRUTH_BOOT = 0.05, (100 if QUICK else 1000)
RECORDING = ["CholCheck"]
OUT = li.RESULTS / "_quick" if QUICK else li.RESULTS
OUT.mkdir(exist_ok=True)
def save(d, name):
    return li.save(d, name, folder=OUT, show=0)

DS = {
    "nhanes": dict(label="diabetes_recorded", truth="diabetes_true", nb="08", tag="nhanes",
                   protected=["Sex", "Age", "race_black", "race_hispanic"],
                   features=["HighBP", "HighChol", "CholCheck", "AnyHealthcare", "RoutineCare", "VisitedDoctor", "BMI",
                             "Smoker", "Stroke", "HeartDiseaseorAttack", "GenHlth", "Sex", "Age", "Education",
                             "Income", "race_black", "race_hispanic"],
                   files=("nhanes_train", "nhanes_val", "nhanes_test")),
    "cdc": dict(label="diabetes_any", truth=None, nb="06", tag="cdcfull", protected=["Sex", "Age"],
                features=list(li.CDC_DIABETES.numeric), files=("cdcfull_train", "cdcfull_val", "cdcfull_test")),
}
for k, d in DS.items():
    d["train"], d["val"], d["test"] = (pd.read_csv(li.DATA / f"{f}.csv") for f in d["files"])
    print(f"{k}: train {len(d['train']):,} rows, {len(d['features'])} features")

nhanes: train 18,463 rows, 17 features
cdc: train 152,208 rows, 21 features


## 2. Bootstrap PC-G² and LiM on the same resamples Stage 0 used
* The discovery sample and the 50 resamples are drawn exactly as in Stage 0 (same seed), so the three
  algorithms see the same rows.
* **PC-G²**: variables with more than 5 distinct values (BMI, age, income, ...) are cut into fifths; the G²
  test then treats every variable as categorical. Full search depth.
* **LiM**: binary variables are passed as 0/1 and marked discrete; the others are standardized and marked
  continuous. The global estimation step only (its local search is too slow for 50 resamples on this many
  rows; disclose this).
* An edge counts for a resample if the two variables are adjacent in either direction.

Run time: Total runtime takes about 2.5 full days running non-stop on a regular laptop using parallel processing; NHANES takes about 14 hours for both PC-G² (which is the longest)
about 27 minutes per PC-G² resample on one core, so with 10 cores roughly
2-2.5 hours per dataset (4-5 hours for both); LiM is much faster. Run it overnight with the laptop plugged in
and sleep turned off. Progress prints after every batch of resamples and is saved as it goes: if the run
stops, just run this cell again and it resumes.
Each finished algorithm-dataset pair is saved at once; with `REUSE = True`, finished pairs are loaded.

In [2]:
REUSE = True
from causallearn.search.ConstraintBased.PC import pc
import lingam
from joblib import Parallel, delayed

def discovery_sample(d):
    cols = [d["label"]] + d["features"]
    x = d["train"][cols].astype(float)
    return (x.sample(MAX_N, random_state=SEED) if len(x) > MAX_N else x).reset_index(drop=True), cols

def adj_pc(x):
    xd = x.iloc[:PC_MAX_N].copy()
    for c in xd.columns:
        if xd[c].nunique() > N_BINS:
            xd[c] = pd.qcut(xd[c].rank(method="first"), N_BINS, labels=False)
        else:
            xd[c] = pd.factorize(xd[c], sort=True)[0]
    G = pc(xd.to_numpy(float), PC_ALPHA, "gsq", show_progress=False).G.graph
    return (G != 0) | (G.T != 0)

def adj_lim(x):
    x = x.iloc[:LIM_MAX_N].copy()
    binary = np.array([x[c].nunique() <= 2 for c in x.columns])
    for c, b in zip(x.columns, binary):
        x[c] = (x[c] == x[c].max()).astype(float) if b else (x[c] - x[c].mean()) / (x[c].std() or 1)
    m = lingam.LiM()
    with contextlib.redirect_stdout(io.StringIO()):          # LiM prints its weight matrix; keep the output clean
        m.fit(x.to_numpy(float), np.array([[0 if b else 1 for b in binary]]), only_global=True)
    A = np.abs(np.asarray(m.adjacency_matrix_)) > 0
    return A | A.T

ALGS = {"PC-G2": adj_pc, "LiM": adj_lim}
EDGES = {}
for k, d in DS.items():
    x, cols = discovery_sample(d)
    d["cols"] = cols
    for alg, fn in ALGS.items():
        path = OUT / f"10_edges_{alg}_{k}.csv"
        if REUSE and path.exists():
            EDGES[(k, alg)] = pd.read_csv(path); print(f"{k} {alg}: loaded {path.name}"); continue
        rng = np.random.default_rng(SEED)                     # same resample sequence as Stage 0's bootstrap
        idxs = [rng.integers(0, len(x), len(x)) for _ in range(BOOT)]
        # checkpoint: resamples are run in batches of N_JOBS and the running total is saved after each batch,
        # so a stopped run (sleep, restart) resumes where it left off
        ckpt = OUT / f"10_checkpoint_{alg}_{k}.npz"
        count, done = np.zeros((len(cols), len(cols))), 0
        if ckpt.exists():
            z = np.load(ckpt); count, done = z["count"], int(z["done"])
            print(f"{k} {alg}: resuming after {done} finished resamples", flush=True)
        t0 = time.time()
        while done < BOOT:
            batch = idxs[done: done + N_JOBS]
            mats = Parallel(n_jobs=N_JOBS)(delayed(fn)(x.iloc[i].reset_index(drop=True)) for i in batch)
            count = count + np.sum(mats, axis=0); done += len(batch)
            np.savez(ckpt, count=count, done=done)
            el = time.time() - t0
            print(f"{k} {alg}: {done}/{BOOT} resamples, {el / 60:.0f} min so far", flush=True)
        prob = count / BOOT
        e = pd.DataFrame([{"a": cols[i], "b": cols[j], "prob_adjacent": prob[i, j]}
                          for i in range(len(cols)) for j in range(i + 1, len(cols)) if prob[i, j] > 0])
        save(e, path.name); EDGES[(k, alg)] = e
    # DirectLiNGAM: the saved Stage 0 bootstrap from notebook 06/08 (unchanged)
    s = pd.read_csv(li.RESULTS / f"{d['nb']}_stage0_edges_{d['tag']}.csv")
    s = s.assign(a=np.minimum(s["from"], s["to"]), b=np.maximum(s["from"], s["to"])).groupby(["a", "b"], as_index=False)["prob_adjacent"].max()
    EDGES[(k, "DirectLiNGAM")] = s
ALG_ORDER = ["DirectLiNGAM", "PC-G2", "LiM"]

nhanes PC-G2: loaded 10_edges_PC-G2_nhanes.csv
nhanes LiM: loaded 10_edges_LiM_nhanes.csv
cdc PC-G2: loaded 10_edges_PC-G2_cdc.csv
cdc LiM: loaded 10_edges_LiM_cdc.csv


In [3]:
def stable_nbrs(e, var):
    s = e[e.prob_adjacent >= EDGE_P]
    return set(s.loc[s.a == var, "b"]) | set(s.loc[s.b == var, "a"])

rows = []
for k, d in DS.items():
    for alg in ALG_ORDER:
        e = EDGES[(k, alg)]
        rows.append({"dataset": k, "algorithm": alg,
                     "linked_to_label": ", ".join(sorted(stable_nbrs(e, d["label"]))),
                     "linked_to_recording": ", ".join(sorted(stable_nbrs(e, RECORDING[0]) - {d["label"]})),
                     "n_stable_edges": int((e.prob_adjacent >= EDGE_P).sum())})
nbr = pd.DataFrame(rows)
save(nbr, "10_neighbors.csv")
pd.set_option("display.max_colwidth", 200)
nbr

wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\10_neighbors.csv  (6 rows x 5 cols)


,dataset,algorithm,linked_to_label,linked_to_recording,n_stable_edges
0,nhanes,DirectLiNGAM,"Age, BMI, CholCheck, Education, GenHlth, HeartDiseaseorAttack, HighBP, HighChol, Sex, Stroke, VisitedDoctor, race_black, race_hispanic","Age, AnyHealthcare, BMI, Education, HighBP, HighChol, Income, RoutineCare, Smoker, VisitedDoctor, race_hispanic",102
1,nhanes,PC-G2,"Age, BMI, CholCheck, Education, GenHlth, HeartDiseaseorAttack, HighBP, HighChol, RoutineCare, VisitedDoctor","Age, AnyHealthcare, BMI, Education, HighBP, HighChol, Income, RoutineCare, VisitedDoctor",88
2,nhanes,LiM,"Age, AnyHealthcare, RoutineCare, VisitedDoctor","AnyHealthcare, HighChol, RoutineCare, VisitedDoctor",34
3,cdc,DirectLiNGAM,"Age, BMI, CholCheck, DiffWalk, GenHlth, HeartDiseaseorAttack, HighBP, HighChol, HvyAlcoholConsump, Income, Sex","Age, AnyHealthcare, BMI, HeartDiseaseorAttack, HighBP, HighChol, Income, PhysHlth",147
4,cdc,PC-G2,"Age, BMI, DiffWalk, GenHlth, HeartDiseaseorAttack, HighBP, HighChol, Income","AnyHealthcare, HighBP, HighChol",97
5,cdc,LiM,"AnyHealthcare, CholCheck, GenHlth","Age, AnyHealthcare, DiffWalk, Education, Fruits, HeartDiseaseorAttack, HvyAlcoholConsump, NoDocbcCost, PhysActivity, Stroke, Veggies",43


**How to read it.** Each row is one algorithm on one dataset. "Linked to label" are the variables the rule
treats as possible causes of the outcome; "linked to recording" are those tied to the cholesterol check. A
variable linked to the recording variable but **not** to the label is one where the O/E check is allowed.

## 4. The same candidate rule and O/E check, per algorithm
Identical to sections 3-4 of notebooks 06/08: groups from protected attributes, the recording variable, its
stable neighbors and the label's stable neighbors; O/E from a model fitted outside the group without the
recording variable and the group's own attribute; valid only for the recording variable or attributes linked
to it and not to the label; flagged if valid and the upper 95% bound is below 1.

In [4]:
def groups_for(attr, df):
    v = df[attr]
    if v.nunique() <= 2:
        return [(f"{attr} == {int(x)}", f"`{attr}` == {x}") for x in sorted(v.unique())]
    lo, hi = v.quantile(0.2), v.quantile(0.8)
    return [(f"{attr} <= {lo:g}", f"`{attr}` <= {lo}"), (f"{attr} >= {hi:g}", f"`{attr}` >= {hi}")]

def keep(m):
    return MIN_SHARE <= m.mean() <= MAX_SHARE and m.sum() >= MIN_N_TRAIN

OE_CACHE = {}
def observed_expected(d, k, query, attr):
    if (k, query) in OE_CACHE:
        return OE_CACHE[(k, query)]
    tr, y = d["train"], d["train"][d["label"]].to_numpy()
    m = tr.eval(query).to_numpy(bool)
    feats = [f for f in d["features"] if f not in set(RECORDING) | {attr}]
    clf = li.fit_model_b(tr.loc[~m, feats].to_numpy(float), y[~m], np.ones((~m).sum()), SEED)
    e = clf.predict_proba(tr.loc[m, feats].to_numpy(float))[:, 1].sum(); o = y[m].sum()
    lo, hi = stats.chi2.ppf(0.025, 2 * o) / 2, stats.chi2.ppf(0.975, 2 * (o + 1)) / 2
    OE_CACHE[(k, query)] = (int(o), e, o / e, lo / e, hi / e)
    return OE_CACHE[(k, query)]

rows = []
for k, d in DS.items():
    for alg in ALG_ORDER:
        e = EDGES[(k, alg)]
        whys = {}
        def tag(a, w): whys.setdefault(a, []).append(w)
        for a in d["protected"]: tag(a, "protected")
        for r in RECORDING:
            tag(r, "recording variable")
            for a in sorted(stable_nbrs(e, r) - {d["label"]}): tag(a, "edge to recording")
        lab_n = stable_nbrs(e, d["label"])
        for a in sorted(lab_n - set(RECORDING)): tag(a, "edge to label")
        for a, w in whys.items():
            valid = "recording variable" in w or ("edge to recording" in w and "edge to label" not in w)
            for name, q in groups_for(a, d["train"]):
                if not keep(d["train"].eval(q).to_numpy(bool)):
                    continue
                o, ex, ratio, lo, hi = observed_expected(d, k, q, a) if valid else (np.nan,) * 5
                rows.append({"dataset": k, "algorithm": alg, "group": name, "query": q, "why": "; ".join(w),
                             "valid": valid, "obs_over_exp": ratio, "ci_low": lo, "ci_high": hi,
                             "flagged": bool(valid and hi < 1)})
flags = pd.DataFrame(rows)
save(flags, "10_flags.csv")
flags[flags.valid].pivot_table(index=["dataset", "group"], columns="algorithm", values="flagged", aggfunc="first") \
    .reindex(columns=ALG_ORDER).replace({True: "FLAG", False: "valid, not flagged"}).fillna("not valid / not a candidate")

wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\10_flags.csv  (98 rows x 10 cols)


algorithm                                         DirectLiNGAM  \
dataset group                                                    
cdc     Age <= 5                   not valid / not a candidate   
        Age >= 11                  not valid / not a candidate   
        AnyHealthcare == 0                                FLAG   
        CholCheck == 0                                    FLAG   
        DiffWalk == 1              not valid / not a candidate   
        Education <= 4             not valid / not a candidate   
        Education >= 6             not valid / not a candidate   
        Fruits == 0                not valid / not a candidate   
        HeartDiseaseorAttack == 1  not valid / not a candidate   
        HvyAlcoholConsump == 1     not valid / not a candidate   
        NoDocbcCost == 1           not valid / not a candidate   
        PhysActivity == 0          not valid / not a candidate   
        PhysHlth >= 5                                     FLAG   
        Stroke == 1                not valid / not a candidate   
        Veggies == 0               not valid / not a candidate   
nhanes  AnyHealthcare == 0                  valid, not flagged   
        CholCheck == 0                                    FLAG   
        HighChol == 1              not valid / not a candidate   
        Income <= 0.97                      valid, not flagged   
        Income >= 4.8                       valid, not flagged   
        RoutineCare == 0                                  FLAG   
        Smoker == 1                                       FLAG   

algorithm                                                PC-G2  \
dataset group                                                    
cdc     Age <= 5                   not valid / not a candidate   
        Age >= 11                  not valid / not a candidate   
        AnyHealthcare == 0                                FLAG   
        CholCheck == 0                                    FLAG   
        DiffWalk == 1              not valid / not a candidate   
        Education <= 4             not valid / not a candidate   
        Education >= 6             not valid / not a candidate   
        Fruits == 0                not valid / not a candidate   
        HeartDiseaseorAttack == 1  not valid / not a candidate   
        HvyAlcoholConsump == 1     not valid / not a candidate   
        NoDocbcCost == 1           not valid / not a candidate   
        PhysActivity == 0          not valid / not a candidate   
        PhysHlth >= 5              not valid / not a candidate   
        Stroke == 1                not valid / not a candidate   
        Veggies == 0               not valid / not a candidate   
nhanes  AnyHealthcare == 0                  valid, not flagged   
        CholCheck == 0                                    FLAG   
        HighChol == 1              not valid / not a candidate   
        Income <= 0.97                      valid, not flagged   
        Income >= 4.8                       valid, not flagged   
        RoutineCare == 0           not valid / not a candidate   
        Smoker == 1                not valid / not a candidate   

algorithm                                                  LiM  
dataset group                                                   
cdc     Age <= 5                                          FLAG  
        Age >= 11                           valid, not flagged  
        AnyHealthcare == 0         not valid / not a candidate  
        CholCheck == 0                                    FLAG  
        DiffWalk == 1                       valid, not flagged  
        Education <= 4                      valid, not flagged  
        Education >= 6                                    FLAG  
        Fruits == 0                         valid, not flagged  
        HeartDiseaseorAttack == 1           valid, not flagged  
        HvyAlcoholConsump == 1                            FLAG  
        NoDocbcCost == 1                    valid,

**How to read it.** One row per group that at least one algorithm allowed the check on. "FLAG" = that
algorithm's version of the rule would send the group for verification. Compare the columns: groups flagged
by all three are robust to the choice of algorithm; groups flagged by only one depend on it.

## 5. NHANES: each algorithm's flags against the blood-test truth (held-out adults)
Same scoring as notebook 08: missed-diagnosis ratio per group on validation + test, short lists compared
with each baseline's top-m groups (m = that algorithm's number of flags), 1,000-resample bootstrap.

In [5]:
d = DS["nhanes"]
ev = pd.concat([d["val"], d["test"]], ignore_index=True)
true_pos = ev[d["truth"]].to_numpy() == 1
missed = true_pos & (ev[d["label"]].to_numpy() == 0)

def miss_ratio(m, w=None):
    w = np.ones(len(m)) if w is None else w
    a = (w * (missed & m)).sum(); b_ = (w * (true_pos & m & ~missed)).sum()
    c = (w * (missed & ~m)).sum(); d_ = (w * (true_pos & ~m & ~missed)).sum()
    r = (a / max(a + b_, 1)) / max(c / max(c + d_, 1), 1e-9)
    p = stats.fisher_exact([[int(a), int(b_)], [int(c), int(d_)]], alternative="greater")[1]
    return r, p

ranks = pd.read_csv(li.RESULTS / "08_rankings.csv")
def q_from_name(name):
    """'GenHlth <= 2 & Income >= 4.8' -> '(`GenHlth` <= 2) & (`Income` >= 4.8)'"""
    parts = [p.strip() for p in name.split(" & ")]
    qs = [f"`{p.split()[0]}` {p.split()[1]} {p.split()[2]}" for p in parts]
    return qs[0] if len(qs) == 1 else " & ".join(f"({q})" for q in qs)
lists = {}
for alg in ALG_ORDER:
    f = flags[(flags.dataset == "nhanes") & (flags.algorithm == alg) & flags.flagged]
    lists[f"causal: {alg}"] = list(f["query"])
    m_ = len(f)
    for base in ["slice search", "confident learning", "anomaly detection"]:
        top = ranks[ranks.method == base].sort_values("rank").group.head(m_)
        lists[f"{base} (top {m_}, vs {alg})"] = [q_from_name(g) for g in top]

masks = {q: ev.eval(q).to_numpy(bool) for L in lists.values() for q in L}
rng2 = np.random.default_rng(SEED)
W = [np.bincount(rng2.integers(0, len(ev), len(ev)), minlength=len(ev)).astype(float) for _ in range(N_TRUTH_BOOT)]
rows = []
for name, L in lists.items():
    if not L:
        rows.append({"list": name, "m": 0}); continue
    pt = [miss_ratio(masks[q]) for q in L]
    boot = [np.mean([miss_ratio(masks[q], w)[0] for q in L]) for w in W]
    rows.append({"list": name, "m": len(L), "groups": "; ".join(q.replace("`", "") for q in L),
                 "truly_elevated": int(sum(p < ALPHA / len(L) for _, p in pt)),
                 "mean_missed_ratio": np.mean([r for r, _ in pt]),
                 "ci_low": np.percentile(boot, 2.5), "ci_high": np.percentile(boot, 97.5)})
truth = pd.DataFrame(rows)
save(truth, "10_truth_shortlists.csv")
truth.round(2)

wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\10_truth_shortlists.csv  (12 rows x 7 cols)


,list,m,groups,truly_elevated,mean_missed_ratio,ci_low,ci_high
0,causal: DirectLiNGAM,3,CholCheck == 0.0; RoutineCare == 0.0; Smoker == 1.0,2,1.83,1.58,2.10
1,"slice search (top 3, vs DirectLiNGAM)",3,GenHlth <= 2; Age <= 32; (GenHlth <= 2) & (Income >= 4.8),1,1.40,1.10,1.70
2,"confident learning (top 3, vs DirectLiNGAM)",3,HeartDiseaseorAttack == 1; Stroke == 1; GenHlth >= 4,0,0.49,0.40,0.59
3,"anomaly detection (top 3, vs DirectLiNGAM)",3,Stroke == 1; HeartDiseaseorAttack == 1; RoutineCare == 0,1,1.04,0.88,1.22
4,causal: PC-G2,1,CholCheck == 0.0,1,2.36,1.97,2.79
5,"slice search (top 1, vs PC-G2)",1,GenHlth <= 2,1,1.65,1.36,1.93
6,"confident learning (top 1, vs PC-G2)",1,HeartDiseaseorAttack == 1,0,0.49,0.32,0.66
7,"anomaly detection (top 1, vs PC-G2)",1,Stroke == 1,0,0.37,0.20,0.55
8,causal: LiM,1,CholCheck == 0.0,1,2.36,1.97,2.79
9,"slice search (top 1, vs LiM)",1,GenHlth <= 2,1,1.65,1.36,1.93


**How to read it.** Same as Figure 3 of the paper, once per algorithm. Each causal row is followed by the
baselines' top lists of the *same length*, so the comparison stays fair when an algorithm flags more or
fewer groups. A mean missed ratio above 1 means the list points toward missed diagnoses.

## 6. Summary across algorithms and datasets

In [6]:
summ = []
for k in DS:
    for alg in ALG_ORDER:
        f = flags[(flags.dataset == k) & (flags.algorithm == alg)]
        row = {"dataset": k, "algorithm": alg, "candidates": len(f), "valid_for_check": int(f.valid.sum()),
               "flagged": ", ".join(f.loc[f.flagged, "group"])}
        if k == "nhanes":
            t = truth[truth.list == f"causal: {alg}"].iloc[0]
            te = t.get("truly_elevated", np.nan)
            row.update({"truly_elevated": f"{int(te) if pd.notna(te) else 0} of {int(t.m)}",
                        "mean_missed_ratio": round(t.get("mean_missed_ratio", np.nan), 2)})
        summ.append(row)
summ = pd.DataFrame(summ)
save(summ, "10_summary.csv")
summ

wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\10_summary.csv  (6 rows x 7 cols)


,dataset,algorithm,candidates,valid_for_check,flagged,truly_elevated,mean_missed_ratio
0,nhanes,DirectLiNGAM,22,6,"CholCheck == 0, RoutineCare == 0, Smoker == 1",2 of 3,1.83
1,nhanes,PC-G2,20,4,CholCheck == 0,1 of 1,2.36
2,nhanes,LiM,10,2,CholCheck == 0,1 of 1,2.36
3,cdc,DirectLiNGAM,16,3,"CholCheck == 0, AnyHealthcare == 0, PhysHlth >= 5",NaN,NaN
4,cdc,PC-G2,14,2,"CholCheck == 0, AnyHealthcare == 0",NaN,NaN
5,cdc,LiM,16,13,"Age <= 5, CholCheck == 0, Education >= 6, HvyAlcoholConsump == 1",NaN,NaN
